# Semana 12: braço exploratório IsoMax+ no R1 (E10), treino e inferência

Segue `prereg/plano_isomax.md`. Treina o R1 (codificador congelado) com a perda IsoMax+ no lugar da entropia cruzada, sementes 1 a 3, com a mesma receita dos treinos confirmatórios, e grava a margem e o escore de distância mínima de cada pixel pontuado na validação, no teste da fonte, na Bolívia e nas 561 janelas do T2. **É exploratório** e não muda nada da rodada confirmatória. **Este notebook não calcula nenhuma métrica.**

| Célula | O que faz |
|---|---|
| 1 | Instala o TerraTorch 1.2.11 |
| 2 | Monta o Drive, clona o repositório, confere o plano, a chave e o manifesto da semana 6 |
| 3 | Confere o backbone |
| 4 | Baixa o treino e a validação do Sen1Floods11 e confere as listas |
| 5 | Definições: modelo, camada IsoMax+, treino e inferência (nada roda aqui) |
| 6 | Conferência do código: o R1 confirmatório da semente 1 tem de reproduzir as margens da Bolívia da semana 6 |
| 7 | Treina as 3 sementes com a IsoMax+ (cerca de 12 minutos cada) e grava a validação |
| 8 | Lê o teste da fonte, a Bolívia e as 561 janelas do T2, com os rótulos conferidos com os da semana 6 |
| 9 | Inferência das 3 sementes nos três conjuntos |
| 10 | Remonta o Drive, confere os arquivos pelo SHA-256 e grava o manifesto |

**Saída** em `resultados/semana12/isomax`: `iso_s<semente>_<conjunto>.npz` (`chaves`, `tamanhos`, `margem` e `mds` em float32, um valor por pixel pontuado, na ordem dos `labels_<conjunto>.npz` da semana 6), `pesos_iso_s<semente>.pt`, `exec_iso_s<semente>_t<tentativa>.json`, `conferencia_codigo.json`, `manifesto_sha256.csv` e `info_isomax.json`.

Abra o Colab na conta do mestrado, **com GPU T4**. Tempo total: cerca de 1 hora e meia. **Se a sessão cair**, rode de novo a partir da célula 1 (ou da 2, se o TerraTorch ainda estiver instalado): o que já foi gravado é pulado. Se alguma célula der erro, pare e guarde a saída inteira.

In [ ]:
# Célula 1 de 10: instalação
!pip install -q "terratorch[peft]==1.2.11" "torchgeo==0.7.1" "peft==0.21.2"
!pip uninstall -y -q torchao

from importlib.metadata import version
for pacote in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy'):
    print(f'{pacote:11s} {version(pacote)}')
assert version('terratorch') == '1.2.11', 'Versão do TerraTorch diferente da fixada.'

import sys
if 'numpy' in sys.modules and sys.modules['numpy'].__version__ != version('numpy'):
    print(f"\nREINICIE A SESSÃO: o NumPy carregado é o {sys.modules['numpy'].__version__} e o instalado é o {version('numpy')}.")
    print('Menu Ambiente de execução > Reiniciar sessão (não precisa excluir o ambiente) e rode a partir da célula 2.')
else:
    print('\nNumPy carregado e instalado coincidem. Pode seguir para a célula 2.')

In [ ]:
# Célula 2 de 10: Drive, repositório, plano, chave e manifesto da semana 6
import os, re, json, time, hashlib, shutil, glob, gc, csv, subprocess, datetime
import numpy as np
import psutil

from google.colab import drive
drive.mount('/content/drive')
MEUDRIVE = '/content/drive/MyDrive'
RAIZ = f'{MEUDRIVE}/Mestrado'
assert os.path.isdir(f'{RAIZ}/modelo_fixado'), 'Pasta Mestrado/modelo_fixado não encontrada. Confira se o Colab está na conta do mestrado.'

S5 = f'{RAIZ}/resultados/semana5'
S2R = f'{RAIZ}/resultados/semana2'
MARGENS6 = f'{RAIZ}/resultados/semana6/margens'
CHAVE6 = f'{RAIZ}/resultados/semana6_chave/chave_codigos.json'
HASHES_S1F11 = f'{RAIZ}/resultados/semana6/entradas_sen1floods11_sha256.csv'
JANELAS = f'{RAIZ}/dados/worldfloods_v2/janelas'
SAIDA = f'{RAIZ}/resultados/semana12/isomax'      # pasta fixa: uma sessão nova continua de onde a anterior parou
LOCAL = '/content/isomax'
LIMITE_RAM_PCT = 85    # cuidado operacional entre execuções; não faz parte do método
assert not os.path.exists(f'{SAIDA}/manifesto_sha256.csv'), (
    'A pasta de saída já tem uma rodada completa (manifesto_sha256.csv). Nada é gravado por cima dela.')
for d in (SAIDA, LOCAL):
    os.makedirs(d, exist_ok=True)
for resto in glob.glob(f'{SAIDA}/parcial_*'):
    os.remove(resto)                                    # cópia interrompida de uma sessão anterior

def sha256_arquivo(caminho):
    h = hashlib.sha256()
    with open(caminho, 'rb') as f:
        for bloco in iter(lambda: f.read(1 << 24), b''):
            h.update(bloco)
    return h.hexdigest()

def esperar(caminho, limite=180):
    t0 = time.time()
    while not os.path.exists(caminho):
        if time.time() - t0 > limite:
            raise FileNotFoundError(f'{caminho} não apareceu no Drive em {limite} s. Pare e guarde esta saída.')
        time.sleep(5)

def trazer(no_drive, sha_esperado, pasta=LOCAL):
    # Copia do Drive para o disco local e confere o SHA-256; se não conferir, remonta o Drive uma vez e copia de novo.
    local = f'{pasta}/{os.path.basename(no_drive)}'
    for tentativa in (1, 2):
        esperar(no_drive)
        if not (os.path.exists(local) and sha256_arquivo(local) == sha_esperado):
            shutil.copyfile(no_drive, local + '.parcial')
            os.replace(local + '.parcial', local)
        sha = sha256_arquivo(local)
        if sha == sha_esperado:
            return local
        if tentativa == 1:
            print(f'   {os.path.basename(no_drive)}: SHA-256 diferente do registro; remontando o Drive e copiando de novo.')
            drive.flush_and_unmount()
            time.sleep(10)
            drive.mount('/content/drive', force_remount=True)
    raise ValueError(f'{no_drive}: SHA-256 {sha[:16]}..., o registro diz {sha_esperado[:16]}... Pare e guarde esta saída.')

def gravar_no_drive(local, nome):
    # Copia um arquivo pronto do disco local para a SAIDA com nome provisório e só então dá o nome final.
    parcial = f'{SAIDA}/parcial_{nome}'
    shutil.copyfile(local, parcial)
    os.replace(parcial, f'{SAIDA}/{nome}')

def gravar_com_sha(local, nome):
    sha = sha256_arquivo(local)
    gravar_no_drive(local, nome)
    with open(f'{SAIDA}/{nome}.sha256', 'w') as f:
        f.write(sha + '\n')
    return sha

def por_de_lado(nome):
    # Arquivo de uma gravação interrompida: fica no Drive com outro nome, nada é apagado.
    marca = time.strftime('%Y%m%dT%H%M%S')
    for sufixo in ('', '.sha256'):
        if os.path.exists(f'{SAIDA}/{nome}{sufixo}'):
            os.replace(f'{SAIDA}/{nome}{sufixo}', f'{SAIDA}/descartado_{marca}_{nome}{sufixo}')

def gravado(nome):
    return os.path.exists(f'{SAIDA}/{nome}') and os.path.exists(f'{SAIDA}/{nome}.sha256')

def carregar_npz(caminho):
    with np.load(caminho, allow_pickle=False) as z:
        return {k: z[k] for k in z.files}

REPO_URL = 'https://github.com/RoseBorges44/prithvi-confidence-shift'
REPO_GIT = '/content/repo'
if not os.path.isdir(REPO_GIT):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_GIT], check=True)
else:
    subprocess.run(['git', '-C', REPO_GIT, 'pull', '-q'], check=True)
commit = subprocess.run(['git', '-C', REPO_GIT, 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
data_plano = subprocess.run(['git', '-C', REPO_GIT, 'log', '--diff-filter=A', '-1', '--format=%cI %h', '--',
                             'prereg/plano_isomax.md'], capture_output=True, text=True, check=True).stdout.strip()
assert data_plano, 'prereg/plano_isomax.md não está no repositório.'

CHAVE = json.load(open(CHAVE6))
assert CHAVE == {'8tjg': 'R1', '4m9f': 'R2', '6wke': 'R3'}, 'A chave da semana 6 não é a registrada em prereg/desvios.md.'
COD_R1 = {r: c for c, r in CHAVE.items()}['R1']

boas = [f for t in (1, 2) if os.path.exists(f := f'{S5}/exec_R1_s1_t{t}.json') and json.load(open(f)).get('situacao') == 'ok']
PESOS_R1_S1 = f'{S5}/pesos_R1_s1.pt'
assert len(boas) == 1 and os.path.exists(PESOS_R1_S1), f'Faltam os arquivos do treino R1 s1 em {S5}.'

MANIFESTO6 = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(f'{MARGENS6}/manifesto_sha256.csv', newline=''))}
assert len(MANIFESTO6) == 64, f'O manifesto da semana 6 tem {len(MANIFESTO6)} arquivos, esperado 64.'
ROTULOS = {}
for conj in ('val', 'teste', 'bolivia', 't2'):
    nome = f'labels_{conj}.npz'
    local = trazer(f'{MARGENS6}/{nome}', MANIFESTO6[nome])
    ROTULOS[conj] = carregar_npz(local)
    os.remove(local)
N_ESPERADO = {'val': 89, 'teste': 90, 'bolivia': 15, 't2': 561}
for conj, n in N_ESPERADO.items():
    assert len(ROTULOS[conj]['chaves']) == n, f'labels_{conj}.npz: {len(ROTULOS[conj]["chaves"])} unidades.'

livre = shutil.disk_usage(MEUDRIVE)[2]
print(f'Espaço livre no Drive: {livre / 1e9:.0f} GB.')
assert livre > 3e9, 'Espaço livre no Drive abaixo de 3 GB.'
print(f'Repositório no commit {commit[:7]}. Plano adicionado em {data_plano}. Saída em {SAIDA.replace(MEUDRIVE, "MyDrive")}')

In [ ]:
# Célula 3 de 10: backbone
from huggingface_hub import hf_hub_download

REPO = 'ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL'
ARQUIVO = 'Prithvi_EO_V2_300M_TL.pt'
REVISAO = '63adbd39c271da4c42f447e69b1a7c91a338cdc9'
SHA_BACKBONE = '3629cedfbb350faafcb0dac902ae0d3c927e25ce8d9e0024aa1276ec66956ddb'

no_drive = f'{RAIZ}/modelo_fixado/backbone/{ARQUIVO}'
if not os.path.exists(no_drive):
    print('Backbone ausente no Drive; baixando na revisão fixada.')
    hf_hub_download(REPO, ARQUIVO, revision=REVISAO, local_dir=os.path.dirname(no_drive))
BACKBONE_PT = f'/content/{ARQUIVO}'
if not (os.path.exists(BACKBONE_PT) and sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE):
    shutil.copy(no_drive, BACKBONE_PT)
assert sha256_arquivo(BACKBONE_PT) == SHA_BACKBONE, 'sha256 do backbone diferente do registrado em modelo.md.'
print(f'Backbone conferido ({SHA_BACKBONE[:8]}…), cópia local em {BACKBONE_PT}')

In [ ]:
# Célula 4 de 10: treino e validação do Sen1Floods11, com as listas conferidas
import urllib.request, concurrent.futures as cf

BUCKET = 'https://storage.googleapis.com/sen1floods11/v1.1/'
DADOS = '/content/sen1floods11'
S2 = f'{DADOS}/v1.1/data/flood_events/HandLabeled/S2Hand'
LB = f'{DADOS}/v1.1/data/flood_events/HandLabeled/LabelHand'
SP = f'{DADOS}/v1.1/splits/flood_handlabeled'
SPLITS_DRIVE = f'{RAIZ}/dados/splits'
for d in (S2, LB, SP, SPLITS_DRIVE):
    os.makedirs(d, exist_ok=True)

SHA_LISTA = {'treino': '4d77d923fcedcf1dca02606ec15acfd2e90238ea06f0b40169b470ca4bd756b5',
             'val': '14a372859fb18b1f7f9791810b7a233c2cd53aaab84f74721f9ac3476ba50504'}
CHAVE_RX = re.compile(r'([A-Za-z-]+_\d+)')
def chave_de(nome):
    achado = CHAVE_RX.search(os.path.basename(str(nome)))
    assert achado, f'Nome de chip sem chave reconhecível: {nome!r}'
    return achado.group(1)

def chaves_csv(nome):
    destino = os.path.join(SPLITS_DRIVE, nome)
    if not os.path.exists(destino):
        urllib.request.urlretrieve(BUCKET + 'splits/flood_handlabeled/' + nome, destino)
    chaves = []
    for linha in open(destino).read().strip().splitlines():
        primeiro = os.path.basename(linha.split(',')[0].strip())
        chave = re.sub(r'_(S1Hand|S2Hand|LabelHand)(\.tif)?$', '', primeiro)
        assert chave and chave != primeiro, f'Linha inesperada em {nome}: {linha!r}'
        chaves.append(chave)
    return chaves

TREINO, VAL = chaves_csv('flood_train_data.csv'), chaves_csv('flood_valid_data.csv')
for papel, chaves in (('treino', TREINO), ('val', VAL)):
    lista = '\n'.join(sorted(f'{c}_LabelHand.tif' for c in chaves)) + '\n'
    assert hashlib.sha256(lista.encode()).hexdigest() == SHA_LISTA[papel], (
        f'Lista de {papel} diferente da registrada em splits/README.md.')
assert not set(TREINO) & set(VAL)
assert sorted(VAL) == sorted(str(c) for c in ROTULOS['val']['chaves']), 'A val difere das chaves de labels_val.npz.'

def baixar_simples(url_e_destino):
    url, destino = url_e_destino
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return 0
    for tentativa in range(4):
        try:
            urllib.request.urlretrieve(url, destino + '.parcial')
            os.replace(destino + '.parcial', destino)
            return 1
        except Exception as erro:
            ultimo = erro
            time.sleep(2 * (tentativa + 1))
    raise RuntimeError(f'Falhou: {url} ({ultimo})')

trabalhos = []
for chave in TREINO + VAL:
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
    trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
t0 = time.perf_counter()
with cf.ThreadPoolExecutor(16) as pool:
    novos = sum(pool.map(baixar_simples, trabalhos))
open(f'{SP}/flood_train_data.txt', 'w').write('\n'.join(TREINO) + '\n')
open(f'{SP}/flood_valid_data.txt', 'w').write('\n'.join(VAL) + '\n')
print(f'{len(trabalhos)} arquivos conferidos, {novos} baixados agora, em {time.perf_counter() - t0:.0f} s. '
      f'Treino {len(TREINO)}, val {len(VAL)}; listas iguais às do repositório.')

In [ ]:
# Célula 5 de 10: definições (nada roda aqui)
import warnings, contextlib, io, logging
import torch
import torch.nn.functional as F
import lightning as L
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from importlib.metadata import version
try:
    from terratorch.tasks import SemanticSegmentationTask
    from terratorch.datamodules import Sen1Floods11NonGeoDataModule
except ModuleNotFoundError as erro:
    raise RuntimeError('TerraTorch não está instalado nesta sessão. Rode a célula 1 e confira que ela imprime terratorch 1.2.11.') from erro

warnings.filterwarnings('ignore', category=UserWarning)
assert torch.cuda.is_available(), 'Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > T4.'
ACELERADOR, PRECISAO, DISPOSITIVO = 'gpu', '16-mixed', 'cuda'
BANDAS = ['BLUE', 'GREEN', 'RED', 'NIR_NARROW', 'SWIR_1', 'SWIR_2']
LR_R1 = 5e-4
LOTE, EPOCAS, VAL_A_CADA, LOTE_INF = 16, 50, 5, 8
LIMITE_MIOU, ALFA = 0.80, 0.10
SEMENTES = (1, 2, 3)            # plano
SEMENTE_RESERVA = 4             # plano: substitui uma semente que falhe duas vezes
ESCALA_ENTROPICA = 10.0         # plano: a do código dos autores, só na perda
TRABALHADORES = min(8, os.cpu_count() or 2)
TAM = 512
IDX_BANDAS = [1, 2, 3, 8, 11, 12]          # B2, B3, B4, B8A, B11, B12 nos 13 canais do Sen1Floods11
ESCALA = np.float32(0.0001)

def datamodule_treino():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE, num_workers=TRABALHADORES, bands=BANDAS,
        train_transform=[A.RandomCrop(height=224, width=224, p=1.0), A.HorizontalFlip(p=0.5),
                         A.VerticalFlip(p=0.5), para_tensor()],
        val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=True, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

def datamodule_inferencia():
    para_tensor = lambda: ToTensorV2(transpose_mask=False)
    return Sen1Floods11NonGeoDataModule(
        data_root=DADOS, batch_size=LOTE_INF, num_workers=0, bands=BANDAS,
        train_transform=[para_tensor()], val_transform=[para_tensor()], test_transform=[para_tensor()],
        drop_last=False, constant_scale=0.0001, no_data_replace=0.0, no_label_replace=-1, use_metadata=False)

def tarefa_r1():
    # Igual ao R1 dos notebooks das semanas 5 e 6.
    args = dict(
        backbone='prithvi_eo_v2_300_tl', backbone_pretrained=True, backbone_ckpt_path=BACKBONE_PT,
        backbone_bands=BANDAS,
        necks=[{'name': 'SelectIndices', 'indices': [5, 11, 17, 23]}, {'name': 'ReshapeTokensToImage'}],
        decoder='UperNetDecoder', decoder_channels=256, decoder_scale_modules=True,
        head_dropout=0.1, num_classes=2, rescale=True)
    t = SemanticSegmentationTask(
        model_factory='EncoderDecoderFactory', model_args=args, loss='ce', ignore_index=-1,
        lr=LR_R1, optimizer='AdamW', optimizer_hparams={'weight_decay': 0.05},
        scheduler='CosineAnnealingLR', scheduler_hparams={'T_max': EPOCAS, 'eta_min': 0},
        freeze_backbone=True, plot_on_val=0)
    t.configure_models()
    return t

class IsoMaxMais(torch.nn.Module):
    """Camada IsoMax+ aplicada a cada pixel, no lugar da convolução 1x1 final da cabeça (prereg/plano_isomax.md).

    Logit da classe j: -|d_s| * ||f/||f|| - p_j/||p_j|||| (distância euclidiana entre vetores normalizados, calculada
    como sqrt(2 - 2 cos), que é a mesma distância). No treino, os logits saem multiplicados pela escala entrópica;
    em avaliação, saem sem ela. Inicialização como no código dos autores: protótipos N(0, 1) e d_s = 1."""

    def __init__(self, canais, classes=2):
        super().__init__()
        self.prototipos = torch.nn.Parameter(torch.empty(classes, canais))
        self.escala_distancia = torch.nn.Parameter(torch.empty(1))
        torch.nn.init.normal_(self.prototipos, mean=0.0, std=1.0)
        torch.nn.init.constant_(self.escala_distancia, 1.0)

    def distancias(self, x):
        with torch.autocast(device_type=x.device.type, enabled=False):
            f = F.normalize(x.float(), dim=1)
            p = F.normalize(self.prototipos.float(), dim=1)
            cos = torch.einsum('bchw,kc->bkhw', f, p)
            return torch.sqrt(torch.clamp(2.0 - 2.0 * cos, min=1e-12))

    def forward(self, x):
        logits = -torch.abs(self.escala_distancia.float()) * self.distancias(x)
        return logits * ESCALA_ENTROPICA if self.training else logits

def trocar_cabeca(modelo):
    """Troca a convolução 1x1 final da cabeça de segmentação pela camada IsoMax+. Devolve a camada nova."""
    seq = modelo.head.head
    final = seq[-1]
    assert isinstance(final, torch.nn.Conv2d) and final.out_channels == 2 and final.kernel_size == (1, 1), (
        f'A última camada da cabeça não é a convolução 1x1 esperada: {final}')
    finais = [m for m in modelo.head.modules() if isinstance(m, torch.nn.Conv2d)]
    assert finais == [final], 'A cabeça tem outras convoluções além da final; o modelo não é o esperado.'
    nova = IsoMaxMais(final.in_channels).to(final.weight.device)
    seq[-1] = nova
    return nova

def tarefa_iso():
    t = tarefa_r1()
    trocar_cabeca(t.model)
    treinaveis = sum(p.numel() for p in t.parameters() if p.requires_grad)
    assert abs(treinaveis / 1e6 - 15.1) <= 0.1, f'IsoMax+: {treinaveis / 1e6:.2f} M treináveis'
    return t

@contextlib.contextmanager
def em_silencio():
    logging.disable(logging.CRITICAL)
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            yield
    finally:
        logging.disable(logging.NOTSET)

def carregar_modelo(tipo, pesos_arquivo):
    """tipo 'ce' (R1 confirmatório) ou 'iso'. Os pesos não têm o codificador, que é o backbone congelado."""
    with em_silencio():
        t = tarefa_iso() if tipo == 'iso' else tarefa_r1()
        t.model.to(DISPOSITIVO)
        pesos = torch.load(pesos_arquivo, map_location=DISPOSITIVO)
        do_modelo = set(t.model.state_dict().keys())
        esperado = {k for k in do_modelo if not k.startswith('encoder.')}
        assert set(pesos.keys()) == esperado, 'Os tensores salvos não coincidem com os do modelo.'
        resultado = t.model.load_state_dict(pesos, strict=False)
        assert not resultado.unexpected_keys and set(resultado.missing_keys) == (do_modelo - esperado)
        modelo = t.model.to(DISPOSITIVO).float().eval()
        for p in modelo.parameters():
            p.requires_grad_(False)
        del t, pesos
    gc.collect()
    return modelo

class Vigia(L.Callback):
    """Guarda a val/mIoU a cada validação, para o treino se a perda ficar não finita e confere a camada IsoMax+."""
    def __init__(self):
        self.curva_val_miou = []
        self.perda_nao_finita = None

    def on_train_start(self, trainer, pl_module):
        assert isinstance(pl_module.model.head.head[-1], IsoMaxMais), 'A camada IsoMax+ não está no modelo em treino.'

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        perda = outputs['loss'] if isinstance(outputs, dict) else outputs
        if torch.is_tensor(perda) and not bool(torch.isfinite(perda).all()) and self.perda_nao_finita is None:
            self.perda_nao_finita = dict(epoca=trainer.current_epoch + 1, lote=int(batch_idx))
            trainer.should_stop = True

    def on_validation_end(self, trainer, pl_module):
        if trainer.sanity_checking:
            return
        valor = trainer.callback_metrics.get('val/mIoU')
        if valor is not None:
            self.curva_val_miou.append([trainer.current_epoch + 1, float(valor)])

def saidas(modelo, x):
    z = modelo(x).output.float()
    assert tuple(z.shape[-2:]) == tuple(x.shape[-2:]), f'Saída {tuple(z.shape)}'
    return (z[:, 1] - z[:, 0]), z.max(dim=1).values

def inferir_val(modelo):
    """Margem e escore de distância mínima de cada pixel pontuado da val, em 32 bits, na ordem de labels_val.npz."""
    dm = datamodule_treino()
    dm.setup('validate')
    ds = dm.val_dataset
    chaves = [chave_de(f) for f in ds.image_files]
    assert chaves == [chave_de(f) for f in ds.segmentation_mask_files], 'Imagens e rótulos da val fora de ordem.'
    carregador = DataLoader(ds, batch_size=LOTE_INF, shuffle=False, num_workers=TRABALHADORES)
    modelo = modelo.to(DISPOSITIVO).float().eval()
    por_chave = {}
    k = 0
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        for lote in carregador:
            imagem = dm.aug({'image': lote['image'].to(DISPOSITIVO), 'mask': lote['mask'].to(DISPOSITIVO)})['image']
            m, d = saidas(modelo, imagem)
            m, d = m.cpu().numpy(), d.cpu().numpy()
            for mi, di, yi in zip(m, d, lote['mask'].numpy()):
                pont = (yi == 0) | (yi == 1)
                por_chave[chaves[k]] = (mi[pont].astype(np.float32), di[pont].astype(np.float32), yi[pont].astype(np.uint8))
                k += 1
    r = ROTULOS['val']
    ordem = [str(c) for c in r['chaves']]
    assert sorted(por_chave) == sorted(ordem)
    margem = np.concatenate([por_chave[c][0] for c in ordem])
    mds = np.concatenate([por_chave[c][1] for c in ordem])
    rotulo = np.concatenate([por_chave[c][2] for c in ordem])
    assert np.array_equal(np.array([por_chave[c][0].size for c in ordem]), r['tamanhos']), 'Val: pixels pontuados diferentes da semana 6.'
    assert np.array_equal(rotulo, r['rotulo']), 'Val: rótulos diferentes dos da semana 6.'
    return margem, mds, rotulo

def inferir_conjunto(modelo, conj):
    """Margem e escore de distância mínima de cada pixel pontuado, chips inteiros, 32 bits (código da semana 6)."""
    bruto, ys = conj['bruto'], conj['ys']
    margens, distancias = [], []
    with torch.no_grad(), torch.autocast(device_type=DISPOSITIVO, enabled=False):
        for i in range(0, len(ys), LOTE_INF):
            x = torch.from_numpy(bruto[i:i + LOTE_INF].astype(np.float32) * ESCALA).to(DISPOSITIVO)
            y = torch.from_numpy(ys[i:i + LOTE_INF].astype(np.int64)).to(DISPOSITIVO)
            x = DM_INF.aug({'image': x, 'mask': y})['image']
            assert x.dtype == torch.float32 and x.ndim == 4 and x.shape[1] == len(BANDAS), tuple(x.shape)
            m, d = saidas(modelo, x)
            m, d = m.cpu().numpy(), d.cpu().numpy()
            for j in range(x.shape[0]):
                pont = ys[i + j] >= 0
                margens.append(m[j][pont].astype(np.float32))
                distancias.append(d[j][pont].astype(np.float32))
    return np.concatenate(margens), np.concatenate(distancias)

def miou_binario(pred, y):
    ious = []
    for c in (0, 1):
        tp = np.sum((pred == c) & (y == c)); fp = np.sum((pred == c) & (y != c)); fn = np.sum((pred != c) & (y == c))
        ious.append(tp / (tp + fp + fn))
    return float(np.mean(ious))

# Leitura dos chips do teste e da Bolívia (código da semana 6)
import rasterio
hashes = {l['arquivo']: l['sha256'] for l in csv.DictReader(open(HASHES_S1F11, newline=''))}

def baixar_conferido(trabalho):
    url, destino = trabalho
    nome = os.path.basename(destino)
    if not (os.path.exists(destino) and sha256_arquivo(destino) == hashes[nome]):
        for tentativa in range(4):
            try:
                urllib.request.urlretrieve(url, destino + '.parcial')
                os.replace(destino + '.parcial', destino)
                break
            except Exception:
                time.sleep(2 * (tentativa + 1))
    assert sha256_arquivo(destino) == hashes[nome], f'{nome} diferente do hash gravado na semana 6.'
    return nome

def preencher(x, valor):
    x = np.ma.filled(x.astype(np.float32), np.nan)
    return np.where(np.isnan(x), np.float32(valor), x).astype(np.float32)

def ler_chip(chave):
    with rasterio.open(f'{S2}/{chave}_S2Hand.tif') as s:
        assert s.count == 13 and (s.height, s.width) == (TAM, TAM), f'{chave}: imagem'
        x = preencher(s.read(masked=True), 0.0)[IDX_BANDAS]
    with rasterio.open(f'{LB}/{chave}_LabelHand.tif') as s:
        assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{chave}: rótulo'
        y = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    return x, y

def conferir_rotulos(conj, ys):
    r = ROTULOS[conj]
    pont = ys >= 0
    assert np.array_equal(pont.reshape(len(ys), -1).sum(1), r['tamanhos']), f'{conj}: pixels pontuados diferentes dos da semana 6.'
    assert np.array_equal(ys[pont].astype(np.uint8), r['rotulo']), f'{conj}: rótulos diferentes dos da semana 6.'

def ler_sen1floods11(conj):
    chaves = [str(c) for c in ROTULOS[conj]['chaves']]
    trabalhos = []
    for chave in chaves:
        trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/S2Hand/{chave}_S2Hand.tif', f'{S2}/{chave}_S2Hand.tif'))
        trabalhos.append((f'{BUCKET}data/flood_events/HandLabeled/LabelHand/{chave}_LabelHand.tif', f'{LB}/{chave}_LabelHand.tif'))
    with cf.ThreadPoolExecutor(16) as pool:
        list(pool.map(baixar_conferido, trabalhos))
    X = np.empty((len(chaves), len(IDX_BANDAS), TAM, TAM), np.float32)
    Y = np.empty((len(chaves), TAM, TAM), np.int16)
    for i, c in enumerate(chaves):
        X[i], Y[i] = ler_chip(c)
    conferir_rotulos(conj, Y)
    return dict(chaves=chaves, bruto=X, ys=Y)

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DM_INF = datamodule_inferencia()
CONJ = {}
print('Definições carregadas. Nada foi treinado nem inferido ainda.')

In [ ]:
# Célula 6 de 10: conferência do código com o R1 confirmatório da semente 1
LIMITE_MARGEM = 1e-3
ARQ_CONF = f'{SAIDA}/conferencia_codigo.json'
if os.path.exists(ARQ_CONF):
    print('Conferência do código já feita:', json.load(open(ARQ_CONF))['diferenca_maxima_bolivia'])
else:
    if 'bolivia' not in CONJ:
        CONJ['bolivia'] = ler_sen1floods11('bolivia')
    modelo = carregar_modelo('ce', PESOS_R1_S1)
    with em_silencio():
        m_bol, _ = inferir_conjunto(modelo, CONJ['bolivia'])
    nome6 = f'run_{COD_R1}_s1_bolivia.npz'
    local6 = trazer(f'{MARGENS6}/{nome6}', MANIFESTO6[nome6])
    z6 = carregar_npz(local6)
    os.remove(local6)
    assert m_bol.size == z6['margem'].size, 'Margens da Bolívia com tamanho errado.'
    dif = float(np.max(np.abs(m_bol - z6['margem'])))
    assert dif <= LIMITE_MARGEM, (f'O R1 da semente 1 difere da semana 6 em {dif:.2e} na Bolívia. '
                                  'Nada foi gravado. Pare e guarde esta saída.')
    # A troca da cabeça num modelo novo: só a última camada muda e os parâmetros treináveis ficam em 15,1 M
    with em_silencio():
        t = tarefa_iso()
    nova = t.model.head.head[-1]
    rec = {'diferenca_maxima_bolivia': dif, 'limite': LIMITE_MARGEM, 'pesos_r1_s1_sha256': sha256_arquivo(PESOS_R1_S1),
           'camada_isomax': {'canais': int(nova.prototipos.shape[1]), 'classes': int(nova.prototipos.shape[0])},
           'treinaveis_iso': int(sum(p.numel() for p in t.parameters() if p.requires_grad)),
           'gpu': torch.cuda.get_device_name(0), 'commit_repositorio': commit}
    json.dump(rec, open(ARQ_CONF, 'w'), indent=2)
    del modelo, t
    gc.collect(); torch.cuda.empty_cache()
    print(f'R1 da semente 1 reproduz as margens da Bolívia da semana 6 (diferença máxima {dif:.1e}). '
          f'Camada IsoMax+ com {rec["camada_isomax"]["canais"]} canais; {rec["treinaveis_iso"] / 1e6:.2f} M parâmetros treináveis.')

In [ ]:
# Célula 7 de 10: treino das sementes 1 a 3 com a IsoMax+ e inferência na validação
def caminhos(semente, tentativa=None):
    return dict(registro=f'{SAIDA}/exec_iso_s{semente}_t{tentativa}.json' if tentativa else None,
                val=f'iso_s{semente}_val.npz', pesos=f'pesos_iso_s{semente}.pt')

def concluida(semente):
    p = caminhos(semente)
    for tentativa in (1, 2):
        r = caminhos(semente, tentativa)['registro']
        if os.path.exists(r) and json.load(open(r)).get('situacao') == 'ok':
            return gravado(p['val']) and gravado(p['pesos'])
    return False

def executar(semente, tentativa):
    p = caminhos(semente, tentativa)
    L.seed_everything(semente, workers=True)
    with em_silencio():
        t = tarefa_iso()
    dm, vigia = datamodule_treino(), Vigia()
    trainer = L.Trainer(accelerator=ACELERADOR, devices=1, precision=PRECISAO, max_epochs=EPOCAS,
                        check_val_every_n_epoch=VAL_A_CADA, num_sanity_val_steps=0, logger=False,
                        enable_checkpointing=False, enable_progress_bar=False, enable_model_summary=False,
                        callbacks=[vigia])
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    trainer.fit(t, datamodule=dm)
    minutos = (time.perf_counter() - t0) / 60
    camada = t.model.head.head[-1]
    assert isinstance(camada, IsoMaxMais), 'A camada IsoMax+ não está no modelo treinado.'
    base = dict(braco='IsoMax+', regime='R1', semente=semente, tentativa=tentativa, lr=LR_R1, epocas=EPOCAS, lote=LOTE,
                escala_entropica=ESCALA_ENTROPICA, plano='prereg/plano_isomax.md', plano_adicionado=data_plano,
                gerado_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
                curva_val_miou=vigia.curva_val_miou, minutos=minutos,
                escala_distancia_final=float(camada.escala_distancia.detach().abs().cpu()),
                pico_memoria_gb=torch.cuda.max_memory_allocated() / 1e9, gpu=torch.cuda.get_device_name(0),
                backbone_sha256=SHA_BACKBONE, commit_repositorio=commit,
                versoes={q: version(q) for q in ('terratorch', 'torchgeo', 'peft', 'lightning', 'torch', 'numpy')})
    motivo = None
    final = vigia.curva_val_miou[-1] if vigia.curva_val_miou else None
    if vigia.perda_nao_finita:
        motivo = f"perda não finita na época {vigia.perda_nao_finita['epoca']}, lote {vigia.perda_nao_finita['lote']}"
    elif final is None or final[0] != EPOCAS:
        motivo = 'sem validação na época 50'
    elif not np.isfinite(final[1]):
        motivo = 'val/mIoU não finita na época 50'
    elif final[1] < LIMITE_MIOU:
        motivo = f'val/mIoU {final[1]:.4f} na época 50, abaixo de {LIMITE_MIOU}'
    if motivo:
        base.update(situacao='falhou', motivo=motivo)
        json.dump(base, open(p['registro'], 'w'), indent=2, ensure_ascii=False)
        print(f'IsoMax+ semente {semente}, tentativa {tentativa}: FALHOU ({motivo}).')
        del t, trainer, dm
        gc.collect(); torch.cuda.empty_cache()
        return base

    margem, mds, rotulo = inferir_val(t.model)
    assert np.isfinite(margem).all() and np.isfinite(mds).all(), 'Saídas não finitas na validação: nada foi gravado.'
    pred = (margem > 0).astype(np.uint8)
    miou_32, taxa_erro = miou_binario(pred, rotulo), float((pred != rotulo).mean())
    r = ROTULOS['val']
    por_de_lado(p['val'])
    np.savez_compressed(f'{LOCAL}/{p["val"]}', chaves=r['chaves'], tamanhos=r['tamanhos'], margem=margem, mds=mds)
    sha_val = gravar_com_sha(f'{LOCAL}/{p["val"]}', p['val'])
    estado = {k: v.detach().cpu() for k, v in t.model.state_dict().items() if not k.startswith('encoder.')}
    torch.save(estado, f'{LOCAL}/{p["pesos"]}')
    por_de_lado(p['pesos'])
    sha_pesos = gravar_com_sha(f'{LOCAL}/{p["pesos"]}', p['pesos'])
    base.update(situacao='ok', val_miou_epoca_50=final[1], miou_val_32bits=miou_32, taxa_erro_val=taxa_erro,
                alfa_maior_que_taxa_de_erro=bool(ALFA > taxa_erro), pixels_pontuados=int(r['tamanhos'].sum()),
                sha256={'val': sha_val, 'pesos': sha_pesos}, pesos_sem_codificador=True)
    json.dump(base, open(p['registro'], 'w'), indent=2, ensure_ascii=False)
    print(f'IsoMax+ semente {semente}, tentativa {tentativa}: val/mIoU na época 50 {final[1]:.4f}, mIoU em 32 bits {miou_32:.4f}, '
          f'erro na val {100 * taxa_erro:.2f}%, |d_s| final {base["escala_distancia_final"]:.3f}, {minutos:.1f} min')
    if not base['alfa_maior_que_taxa_de_erro']:
        print(f'   Atenção: α = {ALFA} não é maior que a taxa de erro na val. Relatar.')
    del t, trainer, dm, margem, mds, rotulo
    gc.collect(); torch.cuda.empty_cache()
    return base

EXECUCOES = list(SEMENTES)    # se uma semente falhar duas vezes, troque-a por SEMENTE_RESERVA aqui (plano)
for semente in EXECUCOES:
    if concluida(semente):
        print(f'IsoMax+ semente {semente}: já feita.')
        continue
    pct = psutil.virtual_memory().percent
    if pct > LIMITE_RAM_PCT:
        raise RuntimeError(f'PAREI: RAM em {pct:.0f}%. Reinicie a sessão e rode da célula 2 em diante; o que está feito é pulado.')
    for tentativa in (1, 2):
        registro = caminhos(semente, tentativa)['registro']
        if os.path.exists(registro) and json.load(open(registro)).get('situacao') == 'falhou':
            print(f'IsoMax+ semente {semente}, tentativa {tentativa}: já registrada como falha.')
            continue
        if executar(semente, tentativa)['situacao'] == 'ok':
            break
    else:
        raise RuntimeError(f'A semente {semente} falhou duas vezes. Pelo plano, a semente {SEMENTE_RESERVA} a substitui: '
                           f'troque {semente} por {SEMENTE_RESERVA} na lista EXECUCOES e rode esta célula de novo.')
print('\nGuarde a saída desta célula.')

In [ ]:
# Célula 8 de 10: teste da fonte, Bolívia e as 561 janelas do T2, com os rótulos conferidos
for conj in ('teste', 'bolivia'):
    if conj not in CONJ:
        CONJ[conj] = ler_sen1floods11(conj)
    print(f'{conj}: {len(CONJ[conj]["chaves"])} chips; rótulos iguais aos da semana 6.')

def ler_tabela(nome, sha_esperado):
    local = trazer(f'{S2R}/{nome}', sha_esperado)
    with open(local, newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))

TAB_JANELAS = ler_tabela('t2_janelas.csv', '3c21970835f2d43cd436be9616c451ebbff5fcea138b7401d3867f0901643b2c')
TAB_NUVEM = ler_tabela('t2_janelas_nuvem050.csv', '3b46e703738042b235b68a12600cfc0c7bf917b3cf5dec7efd8b924a02e132a6')
entram = {r['window_id'] for r in TAB_JANELAS + TAB_NUVEM if r['kept_sensitivity'] == 'True'}
chaves_t2 = [str(c) for c in ROTULOS['t2']['chaves']]
assert set(chaves_t2) == entram and len(chaves_t2) == len(entram), 'As janelas de labels_t2.npz não são as 561 das tabelas.'
X = np.lib.format.open_memmap(f'{LOCAL}/t2.npy', mode='w+', dtype=np.int16, shape=(len(chaves_t2), len(IDX_BANDAS), TAM, TAM))
Y = np.empty((len(chaves_t2), TAM, TAM), np.int16)
t0 = time.time()
for i, jid in enumerate(chaves_t2):
    with rasterio.open(f'{JANELAS}/S2/{jid}.tif') as s:
        assert s.count == 6 and (s.height, s.width) == (TAM, TAM) and s.dtypes[0] == 'int16', f'{jid}: imagem'
        X[i] = s.read()
    with rasterio.open(f'{JANELAS}/label/{jid}.tif') as s:
        assert s.count == 1 and (s.height, s.width) == (TAM, TAM), f'{jid}: rótulo'
        Y[i] = preencher(s.read(1, masked=True), -1.0).astype(np.int16)
    if (i + 1) % 100 == 0:
        print(f'   {i + 1} de {len(chaves_t2)} janelas lidas ({time.time() - t0:.0f} s)')
X.flush()
conferir_rotulos('t2', Y)
CONJ['t2'] = dict(chaves=chaves_t2, bruto=X, ys=Y)
print(f't2: {len(chaves_t2)} janelas lidas em {time.time() - t0:.0f} s; rótulos iguais aos da semana 6.')

In [ ]:
# Célula 9 de 10: inferência das 3 sementes no teste da fonte, na Bolívia e no T2
for semente in EXECUCOES:
    assert concluida(semente), f'A semente {semente} não terminou o treino. Rode a célula 7 antes.'
    faltam = [c for c in ('teste', 'bolivia', 't2') if not gravado(f'iso_s{semente}_{c}.npz')]
    if not faltam:
        print(f'IsoMax+ semente {semente}: já feita.')
        continue
    pct = psutil.virtual_memory().percent
    if pct > LIMITE_RAM_PCT:
        raise RuntimeError(f'PAREI: RAM em {pct:.0f}%. Reinicie a sessão e rode da célula 2 em diante; o que está feito é pulado.')
    t0 = time.time()
    pesos = trazer(f'{SAIDA}/pesos_iso_s{semente}.pt', open(f'{SAIDA}/pesos_iso_s{semente}.pt.sha256').read().strip())
    modelo = carregar_modelo('iso', pesos)
    os.remove(pesos)
    for conj in faltam:
        with em_silencio():
            margem, mds = inferir_conjunto(modelo, CONJ[conj])
        r = ROTULOS[conj]
        assert margem.size == mds.size == int(r['tamanhos'].sum()), f'{conj}: tamanho errado; nada foi gravado.'
        assert np.isfinite(margem).all() and np.isfinite(mds).all(), f'{conj}: saídas não finitas; nada foi gravado.'
        nome = f'iso_s{semente}_{conj}.npz'
        por_de_lado(nome)
        np.savez_compressed(f'{LOCAL}/{nome}', chaves=r['chaves'], tamanhos=r['tamanhos'], margem=margem, mds=mds)
        gravar_com_sha(f'{LOCAL}/{nome}', nome)
        os.remove(f'{LOCAL}/{nome}')
        print(f'   semente {semente} {conj}: {margem.size:,} pixels gravados')
        del margem, mds
    print(f'IsoMax+ semente {semente}: {len(faltam)} conjunto(s) em {(time.time() - t0) / 60:.0f} min.')
    del modelo
    gc.collect(); torch.cuda.empty_cache()
print('\nGuarde a saída desta célula.')

In [ ]:
# Célula 10 de 10: conferência final
print('Gravando o Drive e conferindo de novo; pode levar alguns minutos.')
drive.flush_and_unmount()
time.sleep(10)
drive.mount('/content/drive', force_remount=True)

nomes = [f'iso_s{s}_{c}.npz' for s in EXECUCOES for c in ('val', 'teste', 'bolivia', 't2')] + [f'pesos_iso_s{s}.pt' for s in EXECUCOES]
linhas, ruins = [], []
for nome in nomes:
    esperar(f'{SAIDA}/{nome}')
    assert os.path.exists(f'{SAIDA}/{nome}.sha256'), f'Falta {nome}.sha256. Rode as células 7 e 9 de novo.'
    sha = sha256_arquivo(f'{SAIDA}/{nome}')
    if sha != open(f'{SAIDA}/{nome}.sha256').read().strip():
        ruins.append(nome)
        continue
    linhas.append([nome, os.path.getsize(f'{SAIDA}/{nome}'), sha])
for nome in ruins:
    por_de_lado(nome)
assert not ruins, (f'{len(ruins)} arquivo(s) no Drive diferente(s) do gravado: {ruins}. Foram renomeados para descartado_...; '
                   'rode de novo as células 7 e 9 (só eles serão refeitos) e depois esta.')
with open(f'{SAIDA}/manifesto_sha256.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['arquivo', 'bytes', 'sha256'])
    w.writerows(linhas)
registros = []
for s in EXECUCOES:
    for t in (1, 2):
        if os.path.exists(f := f'{SAIDA}/exec_iso_s{s}_t{t}.json'):
            registros.append(json.load(open(f)))
INFO = {'analise': 'exploratoria E10, braco IsoMax+ no R1 (prereg/plano_isomax.md)', 'plano_adicionado': data_plano,
        'commit_repositorio': commit, 'sementes': list(EXECUCOES), 'escala_entropica': ESCALA_ENTROPICA,
        'conferencia_codigo': json.load(open(ARQ_CONF)), 'execucoes': registros,
        'conjuntos': {c: {'unidades': len(ROTULOS[c]['chaves']), 'pixels': int(ROTULOS[c]['tamanhos'].sum())}
                      for c in ('val', 'teste', 'bolivia', 't2')},
        'margem': 'logit de agua menos logit de terra, sem a escala entropica', 'mds': 'maior dos dois logits (menos a distancia minima escalada)',
        'backbone_sha256': SHA_BACKBONE, 'precisao_inferencia': 'float32',
        'versoes': {q: version(q) for q in ('terratorch', 'torchgeo', 'peft', 'torch', 'numpy')},
        'gpu': torch.cuda.get_device_name(0), 'gerado_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
json.dump(INFO, open(f'{SAIDA}/info_isomax.json', 'w'), indent=2, ensure_ascii=False)
print(f'{len(linhas)} arquivos conferidos depois de remontar o Drive.')
for r in registros:
    if r['situacao'] == 'ok':
        print(f"  semente {r['semente']}: val/mIoU {r['val_miou_epoca_50']:.4f}, |d_s| {r['escala_distancia_final']:.3f}, {r['minutos']:.1f} min")
    else:
        print(f"  semente {r['semente']}, tentativa {r['tentativa']}: FALHOU ({r['motivo']})")
print('Manifesto e info_isomax.json gravados em resultados/semana12/isomax. Nenhuma métrica foi calculada.')